# Классификация изображений: полный разбор
**Курс Machine Learning · преподаватель Ибрахим**

Датасет → обучение → дообучение → метрики → проверка работы.

| шаг | что делаем |
|---|---|
| 1 | датасет к единому размеру, **три выборки** — обучающая, проверочная (15 %), тестовая (**по 2 картинки каждого класса**) |
| 2 | своя свёрточная сеть; обучение, затем **дообучение** на меньшем `lr`; графики и точность по классам |
| 3 | `predict` на тестовой выборке: картинка, **все** вероятности, правильный ответ и ответ сети |

Ячейки **ПОКАЗЫВАЮ** — запустить и посмотреть. Ячейки **ПИШЕМ ВМЕСТЕ** — код, который разбираем строка за строкой.

GPU обязателен: «Среда выполнения → Сменить среду выполнения → GPU».

In [ ]:
import os
import glob
import zipfile
import datetime
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from PIL import Image
import matplotlib.pyplot as plt

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('устройство:', device)
if device == 'cpu':
    print('ВНИМАНИЕ: GPU не включён — обучение будет очень долгим')

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_DIR = '/content/drive/MyDrive/IThub/ML/weights'
except ImportError:
    SAVE_DIR = 'weights'
os.makedirs(SAVE_DIR, exist_ok=True)
print('веса будут лежать в:', SAVE_DIR)

---
# Шаг 1. Датасет и три выборки

Впишите **номер варианта**, который дал преподаватель, и запустите ячейку: она скачает и распакует ваш архив.

In [ ]:
VARIANT = 1

In [ ]:
# ПОКАЗЫВАЮ: скачиваем и распаковываем свой вариант
import gdown

IDS = {
    1: '19BNsUO3Bk2bvPGrXIC_Z0zOhVIRu0YZk',
    2: '1SNhrISvYLo1ZIoHGqqjYWnv0wPUO-v4d',
    3: '19N-zDQDVkdXBo_F5HZXQXp5IFnQDCOBM',
    4: '1t9XaizTpwq8b62yK8_0Ocw-NbLdCUJ38',
    5: '1MK3pFoyq1rANNIiRyvNUlh1icYcMwhiD',
    6: '1B9lomTmR2M-qgf0QJ4ovXHZAtfuB4dbn',
    7: '1ABc8un3Sk85R9LjuMx6Letkivb-iExVl',
    8: '1bgCWMtUIBy0so-3rh1d_7-OXONcDa6YK',
    9: '1w3rm7hkiOVDsr_NrBHlPyM4Ur9eqtPsT',
    10: '10LZ6E6i1igVZo2NInCB1RkUV0-Q4Sj01',
    11: '1cKIrgDg9tA9mklLvRmk8K1f1uq9S7n1I',
    12: '19dYRC_nZU0m7zrxxRUVHtwjcdAFnneYB',
    13: '1tYxb9h_bsQdcZrRbLWjt86Riums9ry09',
    14: '1jQY_Bk98tM5gipPLSzOZzpfaQIfMfFSr',
    15: '1-b9nBVq81zOaGpeWLnVyq9fwWhfnUmC6',
    16: '10xw_yG3KItktDEjOw1UnAmYnQ-IB1c9m',
    17: '1XO78GFb0udao8MwbKEZH-rDV7VQNlj2v',
    18: '1PkdF9ercEurS1WFy1C_qbKjxFJH95fWT',
    19: '1LkjJmG85f-u3VVt3kx5CA20rJZTIqG2S',
    20: '16nnHoelYIgCGjivM1KmqI2o3bQ228Ifc',
    21: '1GzToRaxt1h-0t81eK48VLB3Kv_a-72x7',
    22: '1z7rTOoZHXnfC97oE3iKeRSxSlAaZOPKv',
    23: '1aN0cpP0KGJgNrT-Fzp6k4eJHa37jWwSF',
}

assert VARIANT in IDS, 'впишите номер варианта от 1 до 23 в ячейке выше'
name = f'food-{VARIANT:02d}'
if not os.path.exists(name + '.zip'):
    gdown.download(id=IDS[VARIANT], output=name + '.zip')
if not os.path.exists(name):
    zipfile.ZipFile(name + '.zip').extractall(name)

root = name
while len(os.listdir(root)) == 1:
    root = os.path.join(root, os.listdir(root)[0])
print('папка с классами:', root)
print(sorted(os.listdir(root)))

### Загрузка в массивы
Функция та же, что на занятии про сохранение весов: читает все картинки, приводит к размеру `IMG × IMG`
и складывает в один массив `uint8`. Размер выбираете сами:

| `IMG` | плюсы | минусы |
|---|---|---|
| 64 | быстро | мелкие детали теряются |
| 96 | золотая середина | — |
| 128 | лучше видно блюдо | пикселей в 4 раза больше, чем у 64, — обучение заметно дольше |

In [ ]:
# ПОКАЗЫВАЮ
IMG = 128

def load_folder(root, size):
    classes = sorted(d for d in os.listdir(root)
                     if os.path.isdir(os.path.join(root, d)) and not d.startswith(('.', '_')))
    xs, ys = [], []
    for label, cls in enumerate(classes):
        for path in sorted(glob.glob(os.path.join(root, cls, '*'))):
            img = Image.open(path)
            img.draft('RGB', (size, size))
            img = img.convert('RGB').resize((size, size))
            xs.append(np.array(img))
            ys.append(label)
    x = torch.tensor(np.stack(xs)).permute(0, 3, 1, 2)
    return x, torch.tensor(ys), classes


x_all, y_all, classes = load_folder(root, IMG)
print('x_all:', tuple(x_all.shape), '| классов:', len(classes))
for k, cls in enumerate(classes):
    print(f'  {k}: {cls:25s} {(y_all == k).sum().item()} шт.')

### Задание 1.1 · тестовая выборка
Выберите **случайно по 2 картинки из каждого класса** — это `x_test`, `y_test`. Их сеть не увидит ни при обучении,
ни при выборе лучшей эпохи.

*Должно получиться:* `x_test` формы `(2 · число_классов, 3, IMG, IMG)`, а `torch.bincount(y_test)` — все двойки.

In [ ]:
# ПИШЕМ ВМЕСТЕ 1.1

perm = torch.randperm(len(x_all))
test_idx = []
for k in range(len(classes)):
    test_idx += perm[y_all[perm] == k][:2].tolist()

x_test, y_test = x_all[test_idx], y_all[test_idx]
print('x_test', tuple(x_test.shape), '| по классам:', torch.bincount(y_test).tolist())

### Задание 1.2 · обучающая и проверочная
Из **оставшихся** картинок (все, кроме тестовых) сделайте проверочную выборку — **15 %** от всех, остальное — обучающая.
Напечатайте размеры всех трёх выборок.

*Должно получиться:* `train + val + test = len(x_all)`, доля проверочной — от 0.10 до 0.20.

In [ ]:
# ПИШЕМ ВМЕСТЕ 1.2

taken = set(test_idx)
rest = [i for i in perm.tolist() if i not in taken]
n_val = int(len(x_all) * 0.15)

x_val, y_val = x_all[rest[:n_val]], y_all[rest[:n_val]]
x_train, y_train = x_all[rest[n_val:]], y_all[rest[n_val:]]
print('train', len(x_train), '| val', len(x_val), '| test', len(x_test),
      '| сумма', len(x_train) + len(x_val) + len(x_test), 'из', len(x_all))

### Задание 1.3 · посмотреть на данные
Покажите по **4 картинки каждого класса** из обучающей выборки с подписью-названием класса.

In [ ]:
# ПИШЕМ ВМЕСТЕ 1.3

plt.figure(figsize=(8, 2.2 * len(classes)))
for k in range(len(classes)):
    for j, i in enumerate((y_train == k).nonzero().flatten()[:4]):
        plt.subplot(len(classes), 4, 4 * k + j + 1)
        plt.imshow(x_train[i].permute(1, 2, 0))
        plt.title(classes[k], fontsize=9)
        plt.axis('off')
plt.tight_layout()
plt.show()

### Пачки: `TensorDataset` и `DataLoader`
* `TensorDataset(x, y)` — склеивает картинки и метки в пары: `ds[i]` → `(картинка, метка)`;
* `DataLoader(ds, batch_size=64, shuffle=True)` — раздаёт эти пары **пачками** и перемешивает их каждую эпоху;
  для проверки перемешивать незачем.

Картинки уже в памяти, поэтому `DataLoader` их не перечитывает с диска — обучение быстрое.

In [ ]:
# ПОКАЗЫВАЮ: пачки для обучения и проверки
from torch.utils.data import TensorDataset, DataLoader

train_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=64, shuffle=True)
val_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=256)

xb, yb = next(iter(train_loader))
print('пачек за эпоху:', len(train_loader), '| одна пачка:', tuple(xb.shape), tuple(yb.shape))

---
# Шаг 2. Своя сеть, обучение и дообучение

Инструменты прошлого занятия — `evaluate`, `fit` с журналом `history` и `callback`, `plot_history` — даю готовыми.

In [ ]:
# ПОКАЗЫВАЮ: инструменты прошлого занятия
def evaluate(model, loader):
    model.eval()
    loss_sum, correct, count = 0.0, 0, 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device).float() / 255
            yb = yb.to(device)
            out = model(xb)
            loss_sum += F.cross_entropy(out, yb, reduction='sum').item()
            correct += (out.argmax(dim=1) == yb).sum().item()
            count += len(yb)
    return loss_sum / count, correct / count


def fit(model, epochs, lr, history, callback=None):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    cycle = history[-1]['cycle'] + 1 if history else 1
    for _ in range(epochs):
        model.train()
        loss_sum, correct, count = 0.0, 0, 0
        for xb, yb in train_loader:
            xb = xb.to(device).float() / 255
            yb = yb.to(device)
            flip = torch.rand(len(xb), device=device) < 0.5
            xb = torch.where(flip[:, None, None, None], xb.flip(3), xb)
            out = model(xb)
            loss = F.cross_entropy(out, yb)
            opt.zero_grad()
            loss.backward()
            opt.step()
            loss_sum += loss.item() * len(xb)
            correct += (out.argmax(dim=1) == yb).sum().item()
            count += len(xb)
        val_loss, val_acc = evaluate(model, val_loader)
        epoch = len(history) + 1
        history.append({'epoch': epoch, 'cycle': cycle, 'lr': lr,
                        'train_loss': loss_sum / count, 'train_acc': correct / count,
                        'val_loss': val_loss, 'val_acc': val_acc})
        print(f'эпоха {epoch:2d} | train loss {loss_sum / count:.4f} acc {correct / count:.3f}'
              f' | val loss {val_loss:.4f} acc {val_acc:.3f}')
        if callback:
            callback(model, epoch, val_loss)


def plot_history(history):
    ep = [h['epoch'] for h in history]
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, key, title in [(axes[0], 'loss', 'ошибка (loss)'), (axes[1], 'acc', 'точность')]:
        ax.plot(ep, [h['train_' + key] for h in history], 'o-', label='обучающая')
        ax.plot(ep, [h['val_' + key] for h in history], 'o-', label='проверочная')
        for a, b in zip(history, history[1:]):
            if a['cycle'] != b['cycle']:
                ax.axvline(a['epoch'] + 0.5, color='gray', ls='--')
        ax.set_title(title)
        ax.set_xlabel('эпоха')
        ax.legend()
    plt.show()


weights = SAVE_DIR + f'/{name}-@@dt@@' + '_{epoch:03d}-{val_loss:.6f}.pt'
dt = datetime.datetime.now().strftime('%Y-%m-%d_%H-%M')
weights = weights.replace('@@dt@@', dt)


def checkpoint(model, epoch, val_loss):
    torch.save(model.state_dict(), weights.format(epoch=epoch, val_loss=val_loss))


def load_best(model):
    files = glob.glob(SAVE_DIR + f'/{name}-{dt}_*.pt')
    best = min(files, key=lambda p: float(p[:-3].rsplit('-', 1)[1]))
    model.load_state_dict(torch.load(best))
    print('загружен:', os.path.basename(best), '| проверка: loss %.4f, точность %.3f' % evaluate(model, val_loader))


print('шаблон имени весов:', weights)

### Сеть занятий — так делать НЕ надо
Сеть должна быть **непохожей** на эту:

```python
nn.Conv2d(3, 16, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
nn.Conv2d(16, 32, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 12 * 12, классов)
```

Поменять одно число фильтров — **не считается**. Возьмите **минимум две** идеи из списка:

* два свёрточных слоя подряд в каждом блоке и **4–5 блоков** вместо трёх;
* `nn.BatchNorm2d(каналы)` после свёртки — обучение быстрее и стабильнее;
* `nn.AdaptiveAvgPool2d(1)` вместо `Flatten` перед классификатором — среднее по каждой карте, параметров в `Linear` в сотни раз меньше;
* ветки с разными ядрами и `torch.cat` (функциональный подход);
* обход `x + блок(x)` (residual);
* скрытый полносвязный слой с `Dropout` перед выходом.

### Задание 2.1 · своя сеть
Напишите класс `MyNet(nn.Module)` с `__init__(self, n_classes)` и `forward`. Создайте `model = MyNet(len(classes)).to(device)`,
прогоните **две** нулевые картинки и напечатайте форму выхода и число параметров.

*Должно получиться:* выход `(2, число_классов)`. Параметров — разумно от 100 тысяч до 3 миллионов.

In [ ]:
# ПИШЕМ ВМЕСТЕ 2.1

def block(c_in, c_out):
    return nn.Sequential(
        nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
        nn.MaxPool2d(2),
    )


class MyNet(nn.Module):
    def __init__(self, n_classes):
        super().__init__()
        self.features = nn.Sequential(block(3, 32), block(32, 64), block(64, 128), block(128, 256))
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Dropout(0.3), nn.Linear(256, n_classes))

    def forward(self, x):
        return self.head(self.features(x))


model = MyNet(len(classes)).to(device)
print('выход:', tuple(model(torch.zeros(2, 3, IMG, IMG, device=device)).shape))
print('параметров:', sum(p.numel() for p in model.parameters()))

### Задание 2.2 · первый цикл
Создайте журнал `history = []` и обучите модель с `lr=1e-3` и `callback=checkpoint`. Эпох — сколько успеете:
ориентир 15–25.

*Должно получиться:* строки по эпохам, в `SAVE_DIR` — файлы весов. Точность на проверке заметно выше угадывания
(`1 / число_классов`). Для ориентира: сеть из ключа (≈1.2 млн параметров, `IMG = 128`) на RTX 4060 учится около 5 секунд за эпоху.

In [ ]:
# ПИШЕМ ВМЕСТЕ 2.2

history = []
fit(model, 20, 1e-3, history, callback=checkpoint)

### Задание 2.3 · дообучение
Загрузите **лучшие** веса первого цикла функцией `load_best(model)` и дообучите с `lr=1e-4` (5–8 эпох) в **тот же** `history`.
Хотите третий цикл — снова `load_best` и `lr=1e-5`.

*Должно получиться:* номера эпох продолжаются, в журнале минимум два цикла.

In [ ]:
# ПИШЕМ ВМЕСТЕ 2.3

load_best(model)
fit(model, 8, 1e-4, history, callback=checkpoint)

### Задание 2.4 · метрики
1. Нарисуйте графики `plot_history(history)`.
2. Для каждого цикла напечатайте лучшую эпоху: номер, `lr`, `val_loss`, `val_acc`.
3. Загрузите итоговые лучшие веса `load_best(model)` и напечатайте **точность по каждому классу** на проверочной выборке.

*Должно получиться:* графики с пунктиром между циклами, таблица циклов и строка на каждый класс.

In [ ]:
# ПИШЕМ ВМЕСТЕ 2.4

plot_history(history)

for c in sorted({h['cycle'] for h in history}):
    best = min([h for h in history if h['cycle'] == c], key=lambda h: h['val_loss'])
    print(f"цикл {c}: lr={best['lr']:<7} лучшая эпоха {best['epoch']:2d} | "
          f"val_loss {best['val_loss']:.4f} | val_acc {best['val_acc']:.3f}")

load_best(model)
model.eval()
with torch.no_grad():
    pred_val = torch.cat([model(xb.to(device).float() / 255).argmax(dim=1).cpu()
                          for xb, _ in val_loader])
for k, cls in enumerate(classes):
    mask = y_val == k
    print(f'  {cls:25s} {(pred_val[mask] == k).float().mean().item():.3f}')

---
# Шаг 3. Проверка работы на тестовой выборке

Сеть выдаёт **логиты** — сырые числа любого знака. `F.softmax(логиты, dim=1)` превращает их в **вероятности**:
все от 0 до 1, в сумме по строке — 1.

In [ ]:
# ПОКАЗЫВАЮ: логиты -> вероятности
logits = torch.tensor([[2.0, 1.0, 0.1]])
probs = F.softmax(logits, dim=1)
print('логиты     :', logits)
print('вероятности:', probs.round(decimals=3), '| сумма', round(probs.sum().item(), 4))

### Задание 3.1 · predict
Напишите функцию `predict(model, x)`: переводит модель в режим проверки, делит картинки на 255, отдаёт **вероятности**
формы `[N, число_классов]` на CPU. Примените к `x_test` и напечатайте форму результата и суммы по строкам.

*Должно получиться:* форма `(2 · число_классов, число_классов)`, все суммы — 1.0.

In [ ]:
# ПИШЕМ ВМЕСТЕ 3.1

def predict(model, x):
    model.eval()
    with torch.no_grad():
        return F.softmax(model(x.to(device).float() / 255), dim=1).cpu()


p_test = predict(model, x_test)
print('форма:', tuple(p_test.shape))
print('суммы:', p_test.sum(dim=1).round(decimals=4).tolist())
print(p_test.round(decimals=3))

### Задание 3.2 · показать работу сети
Для **каждой** тестовой картинки нарисуйте строку из двух графиков:

* слева — сама картинка, в заголовке `верно: <класс> | сеть: <класс>`, зелёным, если сеть права, красным — если нет;
* справа — горизонтальные столбики **всех** вероятностей с названиями классов.

*Должно получиться:* `2 · число_классов` строк.

In [ ]:
# ПИШЕМ ВМЕСТЕ 3.2

plt.figure(figsize=(10, 2.2 * len(x_test)))
for i in range(len(x_test)):
    true, guess = y_test[i].item(), p_test[i].argmax().item()
    plt.subplot(len(x_test), 2, 2 * i + 1)
    plt.imshow(x_test[i].permute(1, 2, 0))
    plt.title(f'верно: {classes[true]} | сеть: {classes[guess]}',
              color='green' if true == guess else 'red', fontsize=10)
    plt.axis('off')
    plt.subplot(len(x_test), 2, 2 * i + 2)
    plt.barh(classes, p_test[i])
    plt.xlim(0, 1)
    for k, p in enumerate(p_test[i]):
        plt.text(min(p.item() + 0.02, 0.85), k, f'{p.item():.2f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

### Задание 3.3 · итог на тесте
Напечатайте, сколько тестовых картинок сеть распознала верно: `верно N из M`, и итоговую точность на проверочной выборке.

In [ ]:
# ПИШЕМ ВМЕСТЕ 3.3

right = (p_test.argmax(dim=1) == y_test).sum().item()
print(f'тест: верно {right} из {len(y_test)}')
print('проверочная: loss %.4f, точность %.3f' % evaluate(model, val_loader))